# 01 · Baseline stage 1: train the MSE prediction models

The paper's benchmark is the two-stage "predict then optimize" approach. Stage 1, here, fits a return prediction
model by minimizing mean squared error. Stage 2 (notebook 02) sorts stocks on the predictions and builds portfolios.

Setup, following Section 4.1 and Appendix A.2:
* **Rolling windows** that move one quarter at a time: 39 months of training, 9 months of validation, then predict the
  next quarter. Two trading days before each boundary are dropped, because their labels are realized after it.
* **Models:** Linear and NN1 to NN5, with hidden layers of 32, 16, 8, 4 and 2 units; NNk uses the first k. Inputs pass
  through LayerNorm, and each hidden layer has ReLU, BatchNorm and 5% dropout.
* **Training:** Adam with learning rate 1e-3; early stopping after 10 epochs without improvement in validation MSE,
  keeping the best epoch.

Choices the paper does not report: batch size 10,000; at most 100 epochs; labels winsorized at the 0.5% and 99.5%
quantiles within each day, for training and validation only (evaluation uses raw returns); training in percent units.

`DATA_SOURCE = "synthetic"` runs the whole pipeline on simulated data.
Finished model-window pairs are saved and skipped on reruns, so a long run can be interrupted and resumed.

In [1]:
from pathlib import Path

import pandas as pd

from mlm.data import make_synthetic_panel, panel_features, rolling_windows
from mlm.train import TrainConfig, run_rolling

DATA_SOURCE = "cn"
SOURCES = {
    "cn": dict(panel="D:/ML_Markowitz_data/panel_cn", preds="D:/ML_Markowitz_data/preds/cn"),
    "synthetic": dict(panel="data_synth/panel", preds="preds/synthetic"),
}
PANEL_DIR, PRED_DIR = (Path(SOURCES[DATA_SOURCE][k]) for k in ("panel", "preds"))

MODELS = ["Linear", "NN1", "NN2", "NN3", "NN4", "NN5"]
FIRST_TEST, LAST_TEST = "2014-01-01", "2023-12-31"
FEATURES = None
CFG = TrainConfig()
CFG

TrainConfig(lr=0.001, batch_size=10000, max_epochs=100, patience=10, target_scale=100.0, winsor=(0.005, 0.995), embargo=2, data_on_device=True, use_cuda_graph=False, loss='mse', eps=0.5, weight_decay=0.0, seed=0, device='cuda')

In [2]:
if DATA_SOURCE == "synthetic" and not (PANEL_DIR / "meta.json").exists():
    make_synthetic_panel(PANEL_DIR)

features = FEATURES or panel_features(PANEL_DIR)
windows = rolling_windows(FIRST_TEST, LAST_TEST)
print(f"{len(features)} features, {len(windows)} windows")
pd.DataFrame(windows).set_index("name")

251 features, 40 windows


,train_start,val_start,test_start,test_end
name,,,,
2014Q1,2010-01-01,2013-04-01,2014-01-01,2014-03-31
2014Q2,2010-04-01,2013-07-01,2014-04-01,2014-06-30
2014Q3,2010-07-01,2013-10-01,2014-07-01,2014-09-30
2014Q4,2010-10-01,2014-01-01,2014-10-01,2014-12-31
2015Q1,2011-01-01,2014-04-01,2015-01-01,2015-03-31
2015Q2,2011-04-01,2014-07-01,2015-04-01,2015-06-30
2015Q3,2011-07-01,2014-10-01,2015-07-01,2015-09-30
2015Q4,2011-10-01,2015-01-01,2015-10-01,2015-12-31
2016Q1,2012-01-01,2015-04-01,2016-01-01,2016-03-31


In [3]:
log = run_rolling(MODELS, PANEL_DIR, windows, PRED_DIR, CFG, features)

2014Q1 Linear epochs= 19 val_mse=5.1920 test_R2=+0.1939% (7.1s)


2014Q1 NN1    epochs= 16 val_mse=5.1905 test_R2=+0.2286% (6.1s)


2014Q1 NN2    epochs= 14 val_mse=5.1912 test_R2=+0.2429% (5.5s)


2014Q1 NN3    epochs= 14 val_mse=5.1900 test_R2=+0.2173% (6.1s)


2014Q1 NN4    epochs= 24 val_mse=5.1890 test_R2=+0.2357% (12.5s)


...

2023Q4 NN1    epochs= 13 val_mse=4.0564 test_R2=+0.2519% (14.7s)


2023Q4 NN2    epochs= 11 val_mse=4.0562 test_R2=+0.2796% (14.2s)


2023Q4 NN3    epochs= 11 val_mse=4.0549 test_R2=+0.2270% (16.3s)


2023Q4 NN4    epochs= 12 val_mse=4.0547 test_R2=+0.3180% (19.5s)


2023Q4 NN5    epochs= 20 val_mse=4.0536 test_R2=+0.3356% (35.3s)


## Training summary (averages over windows)

In [4]:
log.groupby("model")[["epochs", "best_epoch", "val_mse", "test_r2", "seconds"]].mean().round(4)

,epochs,best_epoch,val_mse,test_r2,seconds
model,,,,,
Linear,21.875,11.875,6.2362,0.0006,12.0325
NN1,17.750,7.750,6.2334,0.0007,13.5200
NN2,16.600,6.600,6.2327,0.0009,14.9275
NN3,16.350,6.350,6.2325,0.0011,15.8725
NN4,19.725,9.725,6.2317,0.0014,21.0450
NN5,27.075,17.075,6.2320,0.0012,30.8150
